# Support Vector Regression — Exercise 2

**Student:** YOUR NAME  |  **CMSID:** YOUR CMSID

**Dataset:** Combined Cycle Power Plant

**Dataset link:** https://www.kaggle.com/datasets/aagmandeep/combined-cycle-power-plant-dataset-and-prediction

This notebook is a complete implementation of the assigned algorithm. Run cells from top to bottom. Replace the student placeholders before submission.

## 1. Imports

- `numpy` provides numerical arrays and mathematical operations.
- `pandas` stores the tabular data in a DataFrame.
- `matplotlib` and `seaborn` create diagnostic plots.
- `time` measures training cost.
- `train_test_split` creates a leakage-safe holdout set.
- `StandardScaler` standardizes features.
- `SVR` implements epsilon-insensitive kernel regression.
- `GridSearchCV` performs cross-validated hyperparameter search.
- Regression metrics quantify prediction error.

In [ ]:
# Import numerical, tabular, plotting, timing, modelling, and evaluation tools.
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42  # Fixed seed makes the split and comparisons reproducible.

## 2. Load and understand the data

Upload the CSV downloaded from the assigned Kaggle page. The four predictors are ambient temperature, exhaust vacuum, ambient pressure, and relative humidity. The target is net hourly electrical energy output (`PE`). The expected shape is approximately 9,568 rows and four features. The relationships are smooth and nonlinear, which motivates an RBF kernel and an epsilon-insensitive tube.

In [ ]:
# Upload the downloaded CSV in Colab, then set CSV_PATH to its actual name.
from google.colab import files
# uploaded = files.upload()  # Run once, choose the Combined Cycle Power Plant CSV.
CSV_PATH = "Folds5x2_pp.csv"  # Change this if the uploaded filename differs.
df = pd.read_csv(CSV_PATH)

# Normalize common UCI/Kaggle column names so the rest of the notebook is stable.
if set(["AT", "V", "AP", "RH", "PE"]).issubset(df.columns):
    df = df[["AT", "V", "AP", "RH", "PE"]].copy()
else:
    df.columns = ["AT", "V", "AP", "RH", "PE"]

print("shape:", df.shape)
display(df.head())
display(df.describe())
print("missing values:\n", df.isna().sum())

## 3. Exploratory analysis

The histogram checks the target distribution, while correlations give a first linear-strength diagnostic; neither replaces the nonlinear model. A smooth cloud or smooth feature-target relationship is compatible with the RBF kernel. The epsilon tube ignores small residuals, reducing sensitivity to insignificant measurement noise.

In [ ]:
# Plot the target distribution and feature-target correlations.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df["PE"], kde=True, ax=axes[0])
axes[0].set_title("Target distribution: net power output")
correlations = df.corr(numeric_only=True)["PE"].sort_values()
correlations.drop("PE").plot(kind="bar", ax=axes[1], title="Feature-target correlations")
plt.tight_layout()
display(correlations)

## 4. Preprocessing, model, and training procedure

For the RBF kernel, `k(x,x') = exp(-gamma ||x-x'||^2)`. Without scaling, a feature with a larger numerical unit dominates the distance and therefore dominates the kernel. We fit the scaler only on training data to prevent test-set distribution information from entering training.

`C` penalizes violations of the epsilon tube; `epsilon` is the tube half-width; `gamma` controls kernel locality. The pipeline guarantees that every cross-validation fold scales only its training portion.

In [ ]:
# Separate predictors and target, then create a reproducible 80/20 holdout split.
X = df[["AT", "V", "AP", "RH"]]
y = df["PE"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

# The pipeline fits StandardScaler on each training fold, then fits SVR on scaled data.
svr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(kernel="rbf", C=100.0, epsilon=0.1, gamma="scale"))
])

# Search meaningful logarithmic ranges because C and gamma act multiplicatively.
param_grid = {
    "svr__C": [1, 10, 100, 1000],
    "svr__epsilon": [0.01, 0.1, 0.5],
    "svr__gamma": ["scale", 0.001, 0.01, 0.1]
}
search = GridSearchCV(
    svr_pipeline, param_grid, cv=5, scoring="neg_root_mean_squared_error",
    n_jobs=-1, return_train_score=True
)
start = time.perf_counter()
search.fit(X_train, y_train)
fit_seconds = time.perf_counter() - start
model = search.best_estimator_
print("best parameters:", search.best_params_)
print("CV RMSE:", -search.best_score_)
print("fit seconds:", fit_seconds)

### SVR training pipeline

1. The scaler transforms each training feature to `(x - mean)/standard deviation`.
2. The selected `C`, `epsilon`, and `gamma` define the constrained epsilon-SVR objective.
3. The RBF Gram matrix has entries `K_ij = exp(-gamma ||x_i-x_j||²)`.
4. LIBSVM solves the convex quadratic program, so the global optimum is found rather than a local greedy split.
5. Points with nonzero dual weights, including tube violators and boundary points, are support vectors.
6. The intercept `b` is recovered from KKT conditions.
7. Prediction evaluates the kernel expansion `f(x)=sum_i(alpha_i-alpha_i*)K(x_i,x)+b`.

In [ ]:
# Extract the fitted SVR inside the pipeline and inspect its dual representation.
fitted_svr = model.named_steps["svr"]
number_support = len(fitted_svr.support_)
number_training = len(X_train)
print("support vectors:", number_support)
print("support-vector fraction:", number_support / number_training)
print("support_vectors_ shape:", fitted_svr.support_vectors_.shape)
print("dual_coef_ shape:", fitted_svr.dual_coef_.shape)
print("first dual coefficients:", fitted_svr.dual_coef_[0][:10])
print("intercept:", fitted_svr.intercept_)

## 5. Evaluation and interpretation

RMSE is `sqrt(mean((y-yhat)^2))` and penalizes large errors more strongly. MAE is `mean(abs(y-yhat))` and is in target units. R² is `1-SSE/SST`; it measures explained variance relative to predicting the test mean.

In [ ]:
# Generate test predictions and compute all required regression metrics.
y_pred = model.predict(X_test)
rmse = mean_squared_error(y_test, y_pred, squared=False)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE:  {mae:.4f}")
print(f"Test R^2:  {r2:.4f}")

# Plot calibration and residuals to detect systematic error.
residuals = y_test - y_pred
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, y_pred, alpha=0.35)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[0].set(xlabel="Actual", ylabel="Predicted", title="Predicted versus actual")
axes[1].scatter(y_pred, residuals, alpha=0.35)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set(xlabel="Predicted", ylabel="Residual", title="Residual plot")
plt.tight_layout()

In [ ]:
# Measure the C bias-variance trend using the same split and fixed scaling pipeline.
rows = []
for C_value in [0.1, 1, 10, 100, 1000]:
    trial = Pipeline([("scaler", StandardScaler()), ("svr", SVR(kernel="rbf", C=C_value, epsilon=0.1, gamma="scale"))])
    trial.fit(X_train, y_train)
    rows.append({
        "C": C_value,
        "train_RMSE": mean_squared_error(y_train, trial.predict(X_train), squared=False),
        "test_RMSE": mean_squared_error(y_test, trial.predict(X_test), squared=False)
    })
C_results = pd.DataFrame(rows)
display(C_results)
C_results.plot(x="C", y=["train_RMSE", "test_RMSE"], logx=True, marker="o", figsize=(8, 4), title="SVR error versus C")
plt.ylabel("RMSE")
plt.show()

In [ ]:
# Compare fit time for increasing training-set sizes; each subset is scaled inside the pipeline.
time_rows = []
for fraction in [0.25, 0.50, 1.00]:
    n = int(len(X_train) * fraction)
    X_sub, y_sub = X_train.iloc[:n], y_train.iloc[:n]
    timed_model = Pipeline([("scaler", StandardScaler()), ("svr", SVR(kernel="rbf", C=100, epsilon=0.1, gamma="scale"))])
    start = time.perf_counter(); timed_model.fit(X_sub, y_sub); elapsed = time.perf_counter() - start
    time_rows.append({"training_rows": n, "fit_seconds": elapsed})
display(pd.DataFrame(time_rows))

## 6. Reflection

Use the printed numbers to complete this paragraph after running the notebook: increasing `C` usually reduces training bias but can increase variance; `epsilon` controls how many small residuals are ignored; and `gamma` controls locality. The support-vector fraction shows how sparse the kernel expansion is. The timing experiment demonstrates why storing and solving a kernel problem becomes costly as the number of observations grows.